# 第10回 見出しを替えると、クリックは増えるのか（解答例・教員用）


---


## 前回からの接続


第9回は、ペンギンの体の測り方を、楕円の群に分けました。今日は、ニュースの見出しを比べた実験のデータを読みます。見出しを替えると、読む人は増えるのでしょうか。


---


## 今日の分析目標


見出しを替えると、本当にクリックは増えるのか。その差は、偶然でも出る差ではないかに答える。

まず、2つの見出しのクリック率の差と比を出し、表示1回を1つの0/1にします。次に、見出しの札を1万回混ぜて、差が無い世界の差の分布を作り、p値（p-value）を数えます。そして、式の検定（カイ二乗検定・t検定）のp値を、並べ替えと比べます。さらに、手元のデータから戻しながら取り直して、差の区間を作ります。最後に、同じ見出しどうしのテストを何度も比べて、「有意（significant）」の落とし穴を確かめます。

もう1つのねらいは、乱数（random number）で取り直す1回分を`for`でくり返し、結果をリストにためて、分布を作れるようになることです。お手本ごとに、コードを1行ずつ読み（「1行ずつ読む」）、変数の中身を見ます（「中身を見る」）。


### 公開データの世界

来た人をでたらめに2つの組に分け、組ごとに違う見出しを見せて、クリックした人の割合を比べる実験を、A/Bテストといいます。ウェブの会社は、毎日のようにこの実験をしています。

ニュースサイトUpworthyは、2013〜2015年の見出しのA/Bテストを公開しました。研究者がまとめ、論文と一緒に公開しています（The Upworthy Research Archive、CC BY 4.0）。ふつうは社外に出ない実験の記録が、何千件も読めます。


### 場面

編集者は、1つの記事に見出しを何案も作り、どの案を使うかをA/Bテストで決めていました。今日は、そのテストの結果を読み直します。

2つの見出しのクリック率に差があるとき、その差は本物か。本物なら、どれくらいの差か。偶然でも出る差なら、見出しを替える意味はありません。クリックする人は100人に1〜2人で、同じ見出しを2つの組に見せても、クリック率はぴったり同じにはなりません。出てきた差が、偶然の揺れより大きいかを見分けます。


### 手元のデータ

- 4,873件のテストの、22,666の案（見出しなど）。1行が「1つのテストの中の1つの案」
- 列：テストの番号、見出し、表示（見せた回数）、クリック（クリックされた回数）など
- 1つのテストの案は、来た人に、でたらめに割り当てられている

使うときは、論文Matias, Munger, Aubin Le Quere & Ebersole (2021), Scientific Data 8:195を引用します。


### 最後に出す答え

見出しを替えたときのクリック率の差は、偶然でも出る差か。本物なら、差はどれくらいかです。


### 今日の一周（結果だけ先に）

コードの説明はあとにして、今日の解析を、結果だけで一度通して見ておきます。

1. 差を測る：同じ記事の2つの見出しで、Aのクリック率（1.40%）はBの1.67倍です。
2. 並べ替え検定（permutation test）：見出しの札を1万回混ぜて差を測ると、実際の差以上は22回だけです。
3. 式の検定：カイ二乗検定という式で出しても、p値は0.0019。並べ替え（0.0022）とほとんど同じです。
4. ブートストラップ（bootstrap）：手元のデータから取り直すと、差は0.22〜0.92ポイント（95%区間）です。
5. 落とし穴：えっ、まったく同じ見出しの2案で、クリック率が2倍違い、p=0.004で「有意」になります。

このあと、この5つの段の中身を、1つずつ開けていきます。


### 今日の問いとやること

| 段 | 問い | やること |
|---|---|---|
| 差を測る（1節） | どれだけ違う？ | クリック率の差と比を出し、表示1回を1つの0/1にする |
| 並べ替え検定（2節） | 偶然でも出る差か？ | 見出しの札を1万回混ぜて、差の分布を作り、p値を数える |
| 式の検定（3節） | 式でも同じ答え？ | カイ二乗検定とt検定のp値を、並べ替えと比べる |
| ブートストラップ（4節） | 差はどれくらい？ | 戻しながら取り直して、差の区間を作る |
| 落とし穴（5節） | 有意なら本物？ | 同じ見出しどうしのテストを、何度も比べる |


### 今日の進め方

段ごとに、まずスライドで、その段の結論までを通して聞きます。段の終わりにノートへ移り、その段の節のセルを上から実行して、スライドと同じ数が出るかを確かめます。そして、段ごとに1つあるTODOに答えます。

TODOは、実行の前に結果を予想して書き、1か所を替えて実行し、予想と比べる形です。予想が外れても、かまいません。

乱数を使うセルは、どれも最初に`rng = np.random.default_rng(0)`でシードを0に決めています。上から順に実行すれば、スライドと同じ数が出ます。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- クリック率の差と比を出し、割合を0/1の列の平均として扱える
- 札を混ぜて差が無い世界を作り、並べ替え検定でp値を数え、その意味を説明できる
- カイ二乗検定・t検定のp値が、並べ替えのp値とほぼ一致することを確かめられる
- 戻しながら取り直して、差や比の区間を作り、区間と検定の対応を説明できる
- 差が無くても5%は有意になること、比べる組が多いと有意が出やすくなることを説明できる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す。今日は、検定の道具`scipy.stats`も使う
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


---


## 1. どれだけ違う？


ここから、段1「差を測る」（1節）です。問いは、2つの見出しで、クリック率はどれだけ違うか、です。


### お手本：テストの一覧を読む

- 役割：テストの一覧を読み、使う列に日本語の名前を付ける
- 入力：データの置き場所`DATA_DIR`と、縮めた表（`.gz`）
- 出力：表`df`（22,666行）


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
df = pd.read_csv(f'{DATA_DIR}/upworthy_exploratory.csv.gz')
df = df.rename(columns={'clickability_test_id': 'テスト',
                        'headline': '見出し',
                        'impressions': '表示', 'clicks': 'クリック'})
print(len(df), df['テスト'].nunique())


#### 1行ずつ読む

- `DATA_DIR`の行：データの置き場所を`DATA_DIR`にしまう（Colabでは、置き場所のURLが2行に分かれて入っている）
- `read_csv`の行：縮めた表（`.gz`）を読む。`pandas`は、縮めたままの表も読める
- `rename`の3行：使う4本の列に、日本語の名前を付ける（第5回の辞書）
- `print`の行：行の数（案の数）と、テストの番号の種類の数（`nunique`）

4,873件のテストの、22,666の案です。


### お手本：1つのテストを取り出す

- 役割：テストの番号で、1つのテストの行だけを取り出す
- 入力：表`df`
- 出力：表`t`（2行）


In [ ]:
t = df[df['テスト'] == '52d0ad1b1229172bb2000a47']
t[['見出し', '表示', 'クリック']]


#### 1行ずつ読む

- 1行目：テストの番号が一致する行だけを取り出す（第1回の、条件で絞る）
- 2行目：見出しと表示とクリックの列を見る

画像と、見出しの下の短い説明は、2案とも同じです。違うのは見出しだけです。

- A：「ブレインストーミングは最悪。今すぐ、みんなでやめるべき理由」
- B：「次に、これのために職場で会議を開こうとしたら、考え直して」

2014年1月のテストです。どちらも、同じ記事（会議の進め方の話）への入り口です。


### クリック率

案$j$の表示の数を$n_j$、クリックの数を$k_j$とします。クリック率は

$$\hat{p}_j = \frac{k_j}{n_j}$$

です。$\hat{p}$の帽子は「手元のデータから出した値」のしるしです。


### お手本：クリック率


In [ ]:
n = t['表示'].tolist()
k = t['クリック'].tolist()
pA = k[0] / n[0]
pB = k[1] / n[1]
print(n, k)
print(round(pA, 4), round(pB, 4))


#### 1行ずつ読む

- 1・2行目：表示とクリックの列を、リストにする（第4回のリスト）
- 3・4行目：`[0]`がA、`[1]`がB。クリックを表示で割る
- 5・6行目：Aは1.40%、Bは0.84%


### 物差しは2つ：差と比

- 差：$\hat{p}_A - \hat{p}_B$。何ポイント違うか（1ポイント＝1%の差）
- 比：$\hat{p}_A / \hat{p}_B$。何倍か

率が小さいときは、差は小さく見え、比は大きく見えます。両方を見ます。


In [ ]:
print(round(pA - pB, 4))
print(round(pA / pB, 2))


差は0.0056（0.56ポイント）、比は1.67倍です。0.56ポイントは、1万回の表示で56クリックの差です。小さく見えますが、比で言えば1.67倍です。何百万回も表示する見出しなら、大きな違いになります。


### 表示1回を1つの0/1に

表示1回ごとに、クリックされたら1、されなかったら0と書きます。Aは、1が93個、0が6,549個の並びです。こうしておくと、あとで「混ぜる」「取り直す」ができます。


### お手本：0/1の列にする

- 役割：クリックの数だけ1を、残りの数だけ0を並べた列を作る
- 入力：リスト`n`と`k`
- 出力：配列`a`（A）と`b`（B）


In [ ]:
a = np.repeat([1, 0], [k[0], n[0] - k[0]])
b = np.repeat([1, 0], [k[1], n[1] - k[1]])
print(len(a), a.sum(), round(a.mean(), 4))
print(len(b), b.sum(), round(b.mean(), 4))


#### 1行ずつ読む

- 1行目：1を`k[0]`個（93）、0を`n[0] - k[0]`個（6,549）並べる（`np.repeat`）
- 2行目：Bも同じ
- 3・4行目：長さ・1の数・平均


#### 中身を見る：小さな数で

1を2個、0を3個並べてみます。


In [ ]:
small = np.repeat([1, 0], [2, 3])
print(small)
print(small.sum(), small.mean())


和はクリックの数、平均はクリック率です。0/1の列$y_1, \dots, y_n$の平均は

$$\bar{y} = \frac{1}{n}\sum_{i=1}^{n} y_i = \frac{k}{n}$$

で、1の個数を数えて全体で割ったのと同じです。割合の比べ方は、平均の比べ方と同じに扱えます。


### この差は本物か？

同じ見出しを2つの組に見せても、クリック率は少しずれます。0.56ポイントの差は、そのずれより大きいのでしょうか。まだ分かりません。次の段で、偶然のずれを作って比べます。


### 結果を見る前に、予想を書く

同じ見出しを2つの組に見せたら、クリック率はどれくらいずれるでしょうか。予想と、その理由を1行で書いてください。


（ここに予想を書く）


### TODO①：テストを替える（予想 → 実行 → 比べる）

下のセルは、1つのテストの見出しと、案ごとのクリック率を表示します。別のテスト（10代の記事）の2つの見出しは、次の2つです。

- ①「この男の子たちを見ると、10代の男子の頭の中がまったく分からない」
- ②「10代が、ひどい劣等感を笑えるパフォーマンスに変える」

まず、予想を書いてください。どちらの見出しが、よくクリックされるでしょうか。


（ここに予想を書く）


予想を書いたら、セルのテストの番号を`'51c3bde0b95fc097420034ec'`に替えて実行してください。


In [ ]:
# TODO: テストの番号を '51c3bde0b95fc097420034ec' に替えて実行する

tx = df[df['テスト'] == '51c3bde0b95fc097420034ec']
print(tx['見出し'].tolist())
print((tx['クリック'] / tx['表示']).round(4).tolist())


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。どちらが、よくクリックされましたか。


（ここに答えを書く）


①「男の子たち」は1.16%、②「劣等感を笑いに」は1.82%で、②が1.57倍です。見出しを読んで当てるのは、案外むずかしいものです。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 差を測る（1節） | クリック率は1.40%と0.84%。差0.56ポイント、1.67倍 |
| 並べ替え検定（2節） | ？（このあとの段で） |
| 式の検定（3節） | ？（このあとの段で） |
| ブートストラップ（4節） | ？（このあとの段で） |
| 落とし穴（5節） | ？（このあとの段で） |

まだ分からないのは、この差が、偶然でも出る差かどうか、です。


---


## 2. 偶然でも出る差か？


ここから、段2「並べ替え検定」（2節）です。問いは、0.56ポイントの差は、偶然でも出る差か、です。

Aのクリック率は1.40%、Bは0.84%で、差は0.56ポイントでした。表示1回を1つの0/1にし、平均がクリック率になりました。同じ見出しでも、組が違えば率は少しずれます。そのずれの大きさを知りたいのです。


### 差が無い世界

仮に、見出しがクリックに関係ないとします。それなら、クリックする人は、AでもBでもクリックしたはずです。AとBの札は、ただの名札で、付け替えても、クリックの並びは変わりません。

「差が無い」とする仮の考えを、帰無仮説（null hypothesis）といいます。帰無仮説のもとで、札を付け替えた差をたくさん作り、実際の差が、その中でどれくらい珍しいかを見ます。


### 小さな例：8人

Aを見た4人のうち3人、Bを見た4人のうち0人がクリックしたとします。差は0.75です。8枚の札を混ぜ、前の4枚をA、後ろの4枚をBとして、差を測ります。


### お手本：8人で混ぜる

- 役割：8人の0/1を1,000回混ぜ、混ぜるたびの差をリストにためて、p値を数える
- 入力：8人の0/1`x8`
- 出力：差の配列`d8`と、差の大きさが0.75以上だった割合


In [ ]:
x8 = np.array([1, 1, 1, 0, 0, 0, 0, 0])
rng = np.random.default_rng(0)
d8 = []
for i in range(1000):
    y = rng.permutation(x8)
    d8.append(y[:4].mean() - y[4:].mean())
d8 = np.array(d8)
print((np.abs(d8) >= 0.75).mean())


#### 1行ずつ読む

- 1行目：8人の0/1。前の4人がA、後ろの4人がB
- 2行目：乱数を出す道具`rng`を、シード0で作る（同じシードなら、同じ乱数が出る）
- 3行目：差をためる空のリスト（第4回）
- 4・5行目：1,000回くり返す。毎回、8枚を混ぜた並び`y`を作る（`permutation`）
- 6行目：前の4枚の平均 − 後ろの4枚の平均を、リストの後ろに足す
- 7・8行目：配列にして、差の大きさ（`abs`）が0.75以上だった割合


#### 中身を見る：3回だけ

3回だけ混ぜて、並びとリストを表示します。名前は`d8_2`にして、上の`d8`を壊さないようにします。


In [ ]:
rng = np.random.default_rng(0)
d8_2 = []
for i in range(3):
    y = rng.permutation(x8)
    d8_2.append(y[:4].mean() - y[4:].mean())
    print(y, np.array(d8_2))


混ぜるたびに並びが変わり、リストに差が1つずつ増えます。

1,000回のうち147回（14.7%）で、差の大きさが0.75以上でした。見出しがクリックに関係なくても、7回に1回は、これくらいの差が出ます。8人では、3対0でも「偶然では出にくい」とは言えません。


### p値

帰無仮説のもとで、実際の差と同じか、それより大きい差が出る割合を、p値といいます。並べ替えなら

$$p = \frac{|d^{(r)}| \ge |d_{\text{実際}}| \text{ だった回数}}{\text{くり返した回数}}$$

です。$d^{(r)}$は、$r$回目に混ぜたときの差です。


> 今日の重要概念（統計）：p値
>
> p値は、差が無いとしたときに、実際の差と同じかそれ以上の差が、偶然に出る割合。小さければ「偶然では出にくい」と言える。


### 本番：13,452回の表示

8人を、Aの6,642回とBの6,810回の表示に替えます。13,452枚の札（1が150枚）を混ぜ、前の6,642枚をA、残りをBにします。これを1万回くり返します。


### お手本：本番で混ぜる

- 役割：AとBの0/1をつないで1万回混ぜ、混ぜるたびの差をリストにためる
- 入力：配列`a`と`b`
- 出力：差の配列`diffs`、実際の差`obs`


In [ ]:
x = np.concatenate([a, b])
na = len(a)
obs = a.mean() - b.mean()
rng = np.random.default_rng(0)
diffs = []
for i in range(10000):
    y = rng.permutation(x)
    diffs.append(y[:na].mean() - y[na:].mean())
diffs = np.array(diffs)
print(len(x), x.sum(), round(obs, 4))


#### 1行ずつ読む

- 1行目：AとBの0/1を、1本につなぐ（`concatenate`）。札の山
- 2行目：Aの札の数（6,642）
- 3行目：実際の差（0.0056）
- 4〜8行目：8人のときと同じ形。混ぜて、前の`na`枚と残りの差をリストにためる
- 9行目：あとで計算しやすいように、リストを配列にする
- 10行目：札の数13,452、1の数150、実際の差


#### 中身を見る：3回だけ


In [ ]:
rng = np.random.default_rng(0)
diffs2 = []
for i in range(3):
    y = rng.permutation(x)
    diffs2.append(y[:na].mean() - y[na:].mean())
    print(y[:na].sum(), y[na:].sum(), np.array(diffs2).round(4))


150のクリックが、混ぜるたびにAとBに分かれ直します。差は0のまわりです。


### お手本：分布を描く


In [ ]:
plt.hist(diffs, bins=50)
plt.axvline(obs, color='red')
plt.axvline(-obs, color='red', linestyle='--')
plt.xlabel('混ぜたときの差（A − B）')
plt.ylabel('回数')
plt.show()


#### 1行ずつ読む

- 1行目：1万個の差のヒストグラム
- 2行目：実際の差の位置に、赤い縦線（`axvline`）
- 3行目：反対側（Bが0.56ポイント上回る側）にも、点線
- 4〜6行目：軸の名前を付けて表示する

混ぜた差の山は、0のまわりです。実際の差（0.56ポイント）は、山の外れです。


### お手本：p値を数える


In [ ]:
p = (np.abs(diffs) >= obs).mean()
print(p)


#### 1行ずつ読む

- `np.abs(diffs) >= obs`：差の大きさが実際の差以上ならTrue（第6回の比較）
- `.mean()`：Trueを1、Falseを0として平均する。Trueだった割合

8人のときと同じ数え方です。1万回のうち22回だけで、p値は0.0022です。


### 有意

p値が、前もって決めた小さな値（ふつうは0.05）より小さいとき、差は「有意」といいます。この0.05を、有意水準（significance level）といいます。0.0022 < 0.05なので、見出しAとBの差は有意です。

札を混ぜて、差が無い世界の差の分布を作り、p値を数える方法を、並べ替え検定といいます。式を使わず、混ぜて、測って、くり返して、数えます。来た人をでたらめに分けた実験なら、そのまま使えます。


> 重要：p値が小さいのは「偶然では出にくい」ということ。差が大きいこととは別。表示の数が多ければ、小さな差でもp値は小さくなる。


### TODO②：くり返しの回数を替える（予想 → 実行 → 比べる）

下のセルは、札を混ぜて差をためる回数を決め、p値を表示します。

まず、予想を書いてください。100回しか混ぜないと、p値はどうなるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`10000`を`100`に替えて実行してください。


In [ ]:
# TODO: 10000 を 100 に替えて実行する

rng = np.random.default_rng(0)
dx = []
for i in range(100):
    y = rng.permutation(x)
    dx.append(y[:na].mean() - y[na:].mean())
dx = np.array(dx)
print((np.abs(dx) >= obs).mean())


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。p値は、どうなりましたか。


（ここに答えを書く）


100回では、実際の差以上が1回も出ず、p値は0.0になります。「p値が0」ではなく、「100回では足りない」のです。1,000回なら0.002、1万回なら0.0022です。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 差を測る（1節） | クリック率は1.40%と0.84%。差0.56ポイント、1.67倍 |
| 並べ替え検定（2節） | 札を1万回混ぜても、この差以上はめったに出ない（p=0.0022） |
| 式の検定（3節） | ？（このあとの段で） |
| ブートストラップ（4節） | ？（このあとの段で） |
| 落とし穴（5節） | ？（このあとの段で） |

まだ分からないのは、式で出すp値と、並べ替えのp値が合うかどうか、です。


---


## 3. 式でも同じ答え？


ここから、段3「式の検定」（3節）です。問いは、式で出すp値は、並べ替えのp値と合うか、です。

札を1万回混ぜると、実際の差以上は22回で、p値は0.0022でした。0.05より小さいので、差は有意です。混ぜずに、式だけでp値を出す方法もあります。

計算機の無い時代から、差が無い世界の差の分布を、式で求めてきました。割合の差ならカイ二乗検定、平均の差ならt検定です。論文には、こちらがよく出てきます。


### お手本：2×2の表

行が見出し、列がクリックしたかどうかの表です。4つのマスに数を入れます。


In [ ]:
table = [[k[0], n[0] - k[0]],
         [k[1], n[1] - k[1]]]
print(table)


1行目はAのクリックとしなかった数、2行目はBの同じ数です（リストのリスト）。

差が無いなら、AもBも、全体のクリック率（150 / 13,452）でクリックされるはずです。Aで期待されるクリックは

$$6{,}642 \times \frac{150}{13{,}452} = 74.1$$

です。差が無いときに期待される数を、期待度数（expected frequency）といいます。行の和×列の和÷全体です。


### お手本：カイ二乗検定

- 役割：2×2の表から、カイ二乗検定のp値と、期待度数を出す
- 入力：表`table`
- 出力：検定の結果`res`


In [ ]:
res = stats.chi2_contingency(table, correction=False)
print(round(res.pvalue, 4))
print(res.expected_freq.round(1))


#### 1行ずつ読む

- 1行目：2×2の表を渡す（`correction=False`は、小さな数のための補正を外す指定）
- 2行目：p値は0.0019
- 3行目：差が無いときに期待される数（期待度数）


### カイ二乗検定がしていること

4つのマスで、実際の数$O$と期待度数$E$のずれを測ります。

$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

ずれが大きいほど大きくなります。差が無い世界で$\chi^2$がどう散らばるかは、式で分かっています。そこからp値を出します。


#### 中身を見る：4つのマスのずれ


In [ ]:
O = np.array(table)
E = res.expected_freq
print(((O - E) ** 2 / E).round(2))
print(round(res.statistic, 2))


ずれのほとんどは、クリックの列（Aは多すぎ、Bは少なすぎ）から来ます。

並べ替え（1万回）は0.0022、カイ二乗検定は0.0019でした。別のやり方で、ほぼ同じ答えです。表示の数が多いと、混ぜた差の分布は正規分布にほぼ重なります。式はこの正規分布を使うので、答えが合います。


In [ ]:
# @title 図：混ぜた差の分布と正規分布（中身は読み飛ばしてよい）
pbar = (k[0] + k[1]) / (n[0] + n[1])
se0 = np.sqrt(pbar * (1 - pbar) * (1 / n[0] + 1 / n[1]))
plt.figure(figsize=(9, 4))
plt.hist(diffs, bins=50, density=True, color='lightsteelblue')
xs = np.linspace(-0.008, 0.008, 200)
plt.plot(xs, stats.norm.pdf(xs, 0, se0), color='black')
plt.axvline(obs, color='red')
plt.xlabel('混ぜたときの差')
plt.ylabel('密度')
plt.show()


### お手本：t検定

0/1の列の平均を比べるt検定でも、p値を出してみます。


In [ ]:
res_t = stats.ttest_ind(a, b)
print(round(res_t.pvalue, 4))


p値は0.0019です。

t検定は、平均の差を、その揺れの大きさ（標準誤差）で割ります：$t = \dfrac{\bar{a} - \bar{b}}{\text{標準誤差}}$。差が無い世界で$t$がどう散らばるかは、式で分かっています。0/1の平均はクリック率なので、割合の比べ方と同じ答えになります。


### 片側と両側

ここまでのp値は、AとBのどちらが上でも数える両側のp値です。Aが上の側だけ数えるのを、片側といいます。片側のp値は、両側のほぼ半分になります。

「Aのほうが上か」だけを前もって問うなら、片側を使います。データを見てから片側に切り替えるのは、ずるいやり方です。ふつうは両側を使います。


> 重要：カイ二乗検定やt検定は、札を混ぜる代わりに、差が無い世界の分布を式で出す近道。数が多ければ、並べ替えと同じ答えになる。


### TODO③：片側にする（予想 → 実行 → 比べる）

下のセルは、t検定のp値を表示します。`'greater'`は「Aのほうが大きいか」だけを問う片側です。

まず、予想を書いてください。片側にすると、p値はどうなるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`'two-sided'`を`'greater'`に替えて実行してください。


In [ ]:
# TODO: 'two-sided' を 'greater' に替えて実行する

res_x = stats.ttest_ind(a, b, alternative='greater')
print(round(res_x.pvalue, 4))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。p値は、どうなりましたか。


（ここに答えを書く）


片側のp値は0.0009で、両側（0.0019）のほぼ半分です。並べ替えでも、Aが上の側だけ数えると0.0011です（`(diffs >= obs).mean()`）。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 差を測る（1節） | クリック率は1.40%と0.84%。差0.56ポイント、1.67倍 |
| 並べ替え検定（2節） | 札を1万回混ぜても、この差以上はめったに出ない（p=0.0022） |
| 式の検定（3節） | カイ二乗検定でもp=0.0019。並べ替えとほぼ同じ |
| ブートストラップ（4節） | ？（このあとの段で） |
| 落とし穴（5節） | ？（このあとの段で） |

まだ分からないのは、差が本物だとして、どれくらいの大きさか、です。


---


## 4. 差はどれくらい？


ここから、段4「ブートストラップ」（4節）です。問いは、差が本物だとして、どれくらいの大きさか、です。

並べ替えでも、式でも、p値は0.002ほどで、差は有意でした。でも、p値は差の大きさを言いません。0.56ポイントは、どれくらい確かな値でしょうか。

ここで問いが変わります。検定の問いは「差はあるか」（差が無い世界で、どれくらい珍しいか）でした。今度の問いは「差はどれくらいか」（もう一度テストしたら、どれくらいの差が出そうか）です。答えは、1つの値ではなく、幅（区間）で言います。

同じテストを何度もできれば、差がどれくらい揺れるか分かります。でも、テストは1回きりです。そこで、手元のデータを「世界の縮図」とみなして、そこから取り直します。


### 戻しながら取り直す

8枚から1枚引いて記録し、山に戻します。これを8回くり返します。同じ札が2回出ることも、1回も出ないこともあります。

引いた札を戻しながら取り直すことを、復元抽出（sampling with replacement）といいます。手元のデータから復元抽出をくり返し、測った値の揺れを見る方法を、ブートストラップといいます。並べ替えは「混ぜる」、ブートストラップは「戻しながら取り直す」です。


### お手本：8枚から取り直す


In [ ]:
rng = np.random.default_rng(0)
for i in range(3):
    print(rng.choice(x8, size=8))


`choice`は、`x8`から8枚を、戻しながら取ります。1の数が毎回変わります。


### お手本：本番で取り直す

- 役割：AとBをそれぞれ戻しながら取り直し、差をリストにためるのを1万回くり返す
- 入力：配列`a`と`b`
- 出力：差の配列`boot`


In [ ]:
rng = np.random.default_rng(0)
boot = []
for i in range(10000):
    ra = rng.choice(a, size=len(a))
    rb = rng.choice(b, size=len(b))
    boot.append(ra.mean() - rb.mean())
boot = np.array(boot)
print(round(boot.mean(), 4), round(boot.std(), 4))


#### 1行ずつ読む

- 1行目：乱数の道具を、シード0で作る
- 2行目：差をためる空のリスト
- 3行目：1万回くり返す
- 4・5行目：Aの6,642枚から6,642枚、Bの6,810枚から6,810枚を、戻しながら取る
- 6行目：取り直したAとBの差を、リストの後ろに足す
- 7・8行目：配列にして、差の平均（0.0056）と標準偏差（standard deviation）


#### 中身を見る：3回だけ


In [ ]:
rng = np.random.default_rng(0)
boot2 = []
for i in range(3):
    ra = rng.choice(a, size=len(a))
    rb = rng.choice(b, size=len(b))
    boot2.append(ra.mean() - rb.mean())
    print(ra.sum(), rb.sum(), np.array(boot2).round(4))


Aのクリックは93のまわり、Bは57のまわりで揺れます。差も揺れます。


In [ ]:
# @title 図：1回ごとに、後ろに1つ足す（中身は読み飛ばしてよい）
fig, ax = plt.subplots(figsize=(12, 3.6))
ax.axis('off')
for i in range(3):
    yy = 0.8 - i * 0.3
    ax.text(0.0, yy, str(i + 1) + '回目：取り直す → 差 ' + str(round(boot2[i], 4)),
            fontsize=14)
    lst = []
    for v in boot2[:i + 1]:
        lst.append(str(round(v, 4)))
    ax.text(0.5, yy, 'boot = [' + ', '.join(lst) + ']', fontsize=14)
plt.show()


### お手本：分布を描く


In [ ]:
plt.hist(boot, bins=50)
plt.axvline(0, color='gray', linestyle=':')
plt.xlabel('取り直したときの差（A − B）')
plt.ylabel('回数')
plt.show()


### 95%区間

取り直した1万個の差を小さい順に並べると、下から2.5%の値と、上から2.5%の値の間に、真ん中の95%が入ります。この幅を、95%区間といいます（パーセンタイル法）。


In [ ]:
lo, hi = np.percentile(boot, [2.5, 97.5])
print(round(lo, 4), round(hi, 4))


2.5パーセンタイルと97.5パーセンタイル（`percentile`）を、`lo`と`hi`に入れました。差は0.0022〜0.0092、つまり0.22〜0.92ポイントです。

見出しAは、Bより0.2〜0.9ポイントほど、クリック率が高そうです。小さければ0.2ポイント、大きければ0.9ポイントで、幅は広めです。1回のテストで言えるのは、ここまでです。

95%区間が0を含まないことと、両側の検定でp < 0.05であることは、おおよそ対応します。区間は「差があるか」と「どれくらいか」の両方に答えます。論文では、p値と区間を組で書くことが多いです。

並べ替えとブートストラップは、鏡のような関係です。並べ替えは差が無い世界（真ん中は0）を作って、実際の差がどこにあるかを見ます。ブートストラップは手元の世界（真ん中は実際の差）を作って、0がどこにあるかを見ます。


### 式が無い量でも

差の区間は、式でも出せます（標準誤差 × 1.96）。比（何倍か）の区間は、教科書の式が少しややこしくなります。ブートストラップなら、測るものを替えるだけで、同じ手順で出ます。


### お手本：比の区間


In [ ]:
rng = np.random.default_rng(0)
ratios = []
for i in range(10000):
    ra = rng.choice(a, size=len(a))
    rb = rng.choice(b, size=len(b))
    ratios.append(ra.mean() / rb.mean())
print(np.percentile(ratios, [2.5, 97.5]).round(2))


6行目だけが違います（差`-`を比`/`に）。Aは、Bの1.22〜2.36倍です。


### 4案のテスト

ディズニー映画の記事で、見出しだけ違う4案のテストです。

- ①「ディズニーが見せたくない、プリンセス映画の“正しすぎる”別の結末」
- ②「笑：①と同じ」
- ③「『めでたしめでたし』のあとに起きる、不穏な本当の話」
- ④「ディズニー映画は嘘で終わる。その続き」

①が1位でした。でも、もう一度テストしたら、①は1位のままでしょうか。


### お手本：4案を0/1の列に


In [ ]:
d4 = df[df['テスト'] == '5143608a220cb800020009c0']
n4 = d4['表示'].tolist()
k4 = d4['クリック'].tolist()
arms = []
for j in range(4):
    arms.append(np.repeat([1, 0], [k4[j], n4[j] - k4[j]]))
print(n4, k4)


4案の0/1の列を、リスト`arms`にためました。


### お手本：1位が残る割合

- 役割：4案をそれぞれ取り直し、いちばん大きいクリック率の案の番号をためるのを1万回くり返す
- 入力：リスト`arms`
- 出力：1位の番号のリスト`wins`と、番号ごとの割合


In [ ]:
rng = np.random.default_rng(0)
wins = []
for i in range(10000):
    r = []
    for z in arms:
        r.append(rng.choice(z, size=len(z)).mean())
    wins.append(np.argmax(r))
print(pd.Series(wins).value_counts(normalize=True).sort_index())


#### 1行ずつ読む

- 1・2行目：乱数の道具と、1位の番号をためる空のリスト
- 3〜6行目：1万回くり返す。毎回、4案それぞれを取り直し、クリック率をリスト`r`にためる（入れ子の`for`、第9回）
- 7行目：いちばん大きい率の番号（0が①）を、`wins`に足す（`argmax`）
- 8行目：番号ごとの割合

①が1位に残るのは74%です。4回に1回は、②か③が1位になります。「1位の案」にも、揺れがあります。

「何%の確率で1位のままか」を出す、簡単な式はありません。ブートストラップなら、測るもの（いちばん大きい番号）を替えるだけです。取り直して、測って、くり返す。手順は同じです。


> 今日の重要概念（Python）：乱数で取り直して、くり返す
>
> 1回分の「取り直して測る」を`for`で何千回もくり返し、結果をリストにためると、偶然のばらつきの分布ができる。混ぜれば並べ替え検定、戻しながら取れればブートストラップ。


### ブートストラップの限界

- 手元のデータが、世界の縮図になっているときだけ使える
- 数が少ないと、縮図になっていない（8人なら、8枚の中からしか取れない）
- 来た人をでたらめに分けていない比べ方の偏りは、取り直しても直らない


### TODO④：区間の幅を替える（予想 → 実行 → 比べる）

下のセルは、取り直した差の2.5パーセンタイルと97.5パーセンタイル（95%区間）を表示します。`[0.5, 99.5]`にすると、真ん中の99%が入る区間（99%区間）になります。

まず、予想を書いてください。99%区間は、0を含むでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`[2.5, 97.5]`を`[0.5, 99.5]`に替えて実行してください。


In [ ]:
# TODO: [2.5, 97.5] を [0.5, 99.5] に替えて実行する

print(np.percentile(boot, [0.5, 99.5]).round(4))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。99%区間は、0を含みましたか。


（ここに答えを書く）


99%区間は0.0009〜0.0103です。95%より広いですが、まだ0を含みません。p値0.0022は0.01より小さく、99%区間が0を含まないのと対応します。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 差を測る（1節） | クリック率は1.40%と0.84%。差0.56ポイント、1.67倍 |
| 並べ替え検定（2節） | 札を1万回混ぜても、この差以上はめったに出ない（p=0.0022） |
| 式の検定（3節） | カイ二乗検定でもp=0.0019。並べ替えとほぼ同じ |
| ブートストラップ（4節） | 差は0.22〜0.92ポイント（95%区間）、比は1.22〜2.36倍 |
| 落とし穴（5節） | ？（このあとの段で） |

まだ分からないのは、「有意」なら本物と言ってよいか、です。


---


## 5. 有意なら本物？


ここから、段5「落とし穴」（5節）です。問いは、「有意」なら、本物の差と言ってよいか、です。

見出しAとBの差は有意（p=0.0022）で、区間は0.22〜0.92ポイントでした。4案のテストでは、1位が1位に残るのは74%でした。有意という言葉は、どれくらい信じてよいのでしょうか。

Upworthyは、まったく同じ案を2つ以上並べたテストもしていました。見出し・画像・説明・本文の書き出しが、すべて同じです。差が無いと分かっている比べ方で、A/Aテストといいます。


### お手本：投票の記事


In [ ]:
v = df[df['テスト'] == '54592bf482ab36427f000055']
v[['見出し', '表示', 'クリック']]


4案とも同じ見出し「ある男が簡単な計算で、『自分の1票は意味がない』が間違いだと説明する」です。


### 何度も書いたので、関数にする

0/1の列を作る`np.repeat`と、p値を出す`chi2_contingency`を、何度も書きました。この先も何度も使うので、1回分を返す関数にまとめます（第3回の`def`）。中身は、ここまでのセルと同じです。


In [ ]:
def to01(k, n):
    return np.repeat([1, 0], [k, n - k])


def pval(u, w):
    table = [[u.sum(), len(u) - u.sum()],
             [w.sum(), len(w) - w.sum()]]
    return stats.chi2_contingency(table, correction=False).pvalue


#### 1行ずつ読む

- 1・2行目：クリックの数`k`と表示の数`n`から、0/1の列を返す
- 5〜8行目：2つの0/1の列から2×2の表を作り、カイ二乗検定のp値を返す

関数の中の`k`・`n`・`table`は、関数の外の`k`・`n`・`table`とは別物です（第3回）。外のリスト`k`・`n`は壊れません。


### お手本：同じ案どうしを比べる

案1と案3（シェア用の文まで、まったく同じ2案）を比べます。


In [ ]:
nv = v['表示'].tolist()
kv = v['クリック'].tolist()
u = to01(kv[0], nv[0])
w = to01(kv[2], nv[2])
print(round(u.mean(), 4), round(w.mean(), 4))
print(round(pval(u, w), 4))


えっ、同じ案なのに、クリック率は0.65%と1.31%で、2倍です。p値は0.0037で、0.05より小さく「有意」です。差が無いと分かっているのに、有意になりました。


### 道具：同じ案の組を探す

テストの一覧から、同じテストの中で、まったく同じ案の組を探し、組ごとに`pval`でp値を出す道具の関数です。中身は読み飛ばしてかまいません。


In [ ]:
# @title 道具：同じ案の組を探す（中身は読み飛ばしてよい）
def same_pairs(df):
    cols = ['見出し', 'eyecatcher_id', 'excerpt', 'lede', 'share_text',
            'square']
    key = df[cols].astype(str).apply('|'.join, axis=1)
    rows = []
    for (tid, kk), g in df.groupby([df['テスト'], key]):
        if len(g) >= 2:
            s, e = g.iloc[0], g.iloc[1]
            pp = pval(to01(s['クリック'], s['表示']),
                      to01(e['クリック'], e['表示']))
            rows.append([tid, s['表示'], s['クリック'],
                         e['表示'], e['クリック'], pp])
    return pd.DataFrame(rows, columns=['テスト', '表示1', 'クリック1',
                                       '表示2', 'クリック2', 'p'])


In [ ]:
aa = same_pairs(df)
print(len(aa), (aa['p'] < 0.05).sum())


同じ案の51組のうち、2組（3.9%）がp < 0.05です。投票の記事は、その1つです。


### A/Aテストを作る

51組では、少なすぎて割合がはっきりしません。見出しAの6,642回の表示を、でたらめに半分に分ければ、A/Aテストが作れます。これを1,000回くり返し、p < 0.05になる割合を数えます。


In [ ]:
rng = np.random.default_rng(0)
ps = []
for i in range(1000):
    y = rng.permutation(a)
    ps.append(pval(y[:3321], y[3321:]))
ps = np.array(ps)
print((ps < 0.05).mean())


#### 1行ずつ読む

- 1・2行目：乱数の道具と、p値をためる空のリスト
- 3・4行目：1,000回、Aの6,642枚を混ぜる
- 5行目：前の3,321枚と後ろの3,321枚のp値を、リストにためる
- 6・7行目：p < 0.05だった割合


In [ ]:
plt.hist(ps, bins=5, range=(0, 1))
plt.axvline(0.05, color='red')
plt.xlabel('p値')
plt.ylabel('回数')
plt.show()


差が無いとき、p値は0〜1にほぼ平らに散らばります。赤い線（0.05）より左に、約5%が入ります。

有意水準0.05で判定すると、差が無くても、20回に1回は「有意」になります。1,000回のうち61回（6.1%）でした。投票の記事も、その20回に1回だったと考えられます。「有意」は「偶然では出にくい」であって、「本物の証明」ではありません。


### たくさん比べると

4案のテストなら、2案の組は6組あります。6組を全部比べると、どれか1組が「有意」になりやすくなります。たくさんの比べ方から有意なものを探すと、偶然の当たりを拾います。これを、多重比較の問題といいます。


In [ ]:
rng = np.random.default_rng(0)
hit = 0
for i in range(1000):
    g = np.array_split(rng.permutation(a), 4)
    ps6 = []
    for j in range(4):
        for m in range(j + 1, 4):
            ps6.append(pval(g[j], g[m]))
    if min(ps6) < 0.05:
        hit = hit + 1
print(hit / 1000)


#### 1行ずつ読む

- 1〜3行目：乱数の道具と、数える変数`hit`。1,000回くり返す
- 4行目：Aの表示を混ぜて、4つに分ける（4つの同じ案。`array_split`）
- 5〜8行目：6組のp値をリストにためる（入れ子の`for`。`m`は`j`より後ろだけ）
- 9〜11行目：いちばん小さいp値が0.05より小さければ数える

4つの同じ案から6組を比べると、どれか1組が有意になるのは20.3%です。1組だけなら約5%でした。


> 重要：比べる組が多いほど、差が無くても、どれかが「有意」になりやすい。何組を比べたかを、結果と一緒に書く。


### 数が少ないと

表示が少ないと、同じくらいの比の差でも、偶然の揺れに埋もれます。表示が多いと、ごく小さな差でも有意になります。p値は、差の大きさと、数の多さの両方で決まります。


### 無作為に分けたから

A/Bテストは、来た人をでたらめに分けました。組の違いは、見出しだけです。だから、差が偶然でなければ、原因は見出しと言えます。でたらめに分けた実験を、無作為化比較試験といいます。

たとえば、朝は見出しA、夜は見出しBを出したとします。朝と夜では、来る人も、読む時間も違います。差が出ても、見出しのせいか、時間のせいか、分けられません。


### TODO⑤：表示の数を1/10に（予想 → 実行 → 比べる）

下のセルは、ブレインストーミングのテストの表示とクリックを`s`で割り（`//`は、割って小数を切り捨てる計算）、0/1の列にして、比とp値を表示します。

まず、予想を書いてください。同じくらいの比で表示が1/10なら、p値はどうなるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`s = 1`を`s = 10`に替えて実行してください。


In [ ]:
# TODO: 1 を 10 に替えて実行する（表示もクリックも1/10）

s = 10
u = to01(k[0] // s, n[0] // s)
w = to01(k[1] // s, n[1] // s)
print(len(u), len(w), round(u.mean() / w.mean(), 2))
print(round(pval(u, w), 4))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。p値は、どうなりましたか。


（ここに答えを書く）


表示が1/10だと、比は1.85倍でも、p値は0.26で、有意ではありません。差が無いのではなく、数が少なくて見分けられないのです。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 差を測る（1節） | クリック率は1.40%と0.84%。差0.56ポイント、1.67倍 |
| 並べ替え検定（2節） | 札を1万回混ぜても、この差以上はめったに出ない（p=0.0022） |
| 式の検定（3節） | カイ二乗検定でもp=0.0019。並べ替えとほぼ同じ |
| ブートストラップ（4節） | 差は0.22〜0.92ポイント（95%区間）、比は1.22〜2.36倍 |
| 落とし穴（5節） | 同じ見出しどうしでも、約5%は「有意」になる（1,000回で6.1%） |

5つの段がそろい、今日の問いに答えられます。


### 答え

ブレインストーミングの見出しAは、Bよりクリックされました。偶然では出にくい差です（p=0.002）。差は0.2〜0.9ポイント、1.2〜2.4倍ほどです。ただし、同じ見出しどうしでも、約5%は「有意」になります。1回の有意は、本物の証明ではありません。


まだ言えないこと

- なぜAがクリックされたのか（言葉のどこが効いたか）
- ほかの記事の見出しでも、同じ書き方が効くか
- クリックのあと、記事を最後まで読んだか


### レポートなら、こう書く

今日の解析を、期末レポートの項目に写すと、次のようになります。

- データ：Upworthy Research Archive。見出しだけが違う2案のA/Bテスト（6,642回と6,810回の表示）
- 結果：クリック率1.40%と0.84%。差0.56ポイント（95%区間0.22〜0.92）、比1.67倍（1.22〜2.36）
- 検定：並べ替え検定（1万回）でp=0.0022、カイ二乗検定でp=0.0019
- 確かめ：同じ案どうしの比較51組のうち2組がp < 0.05。有意水準どおりの割合
- 解釈：1回のテストの有意は、本物の証明ではない。くり返して確かめる
- 限界：1つの記事の2つの見出し。なぜ効いたか、ほかの記事でも効くかは分からない


---


## 目標に答えられたか


- 今日の目標は「見出しを替えると、本当にクリックは増えるのか。その差は、偶然でも出る差ではないか」でした。p値と区間で答えられますか？
- 1節で、0/1の列の平均が、クリック率になるのはなぜですか？
- 2節で、札を混ぜると「差が無い世界」になるのはなぜですか？
- 3節で、カイ二乗検定と並べ替えのp値がほぼ同じになったのはなぜですか？
- 4節で、並べ替えとブートストラップは、それぞれ何を作りましたか？
- 5節で、同じ見出しどうしなのに有意になったのはなぜですか？


---


## 今日の要点


- 今日、持ち帰る1文：差が偶然でも出るかは、差が無い世界を作って比べる（並べ替え検定・p値）。差がどれくらいかは、手元のデータから取り直して幅で言う（ブートストラップ・区間）
- 重要概念（統計）：p値。差が無いとしたときに、実際の差と同じかそれ以上の差が、偶然に出る割合。差の大きさではないので、区間と組で読む
- 重要概念（Python）：乱数で取り直して、くり返す。1回分の「取り直して測る」を`for`でくり返し、リストにためると、偶然のばらつきの分布ができる
- 式の検定（カイ二乗・t検定）は、並べ替えの近道。数が多ければ同じ答え
- 差が無くても5%は有意になる。比べる組が多いと、さらに増える


---


## 次回へ


今日は、2つの組の差を、偶然と見分けました。では、顔の画像1枚は4,096個の数ですが、誰の顔かを見分けるのに、本当にそれだけの数が要るのでしょうか。


---


## 課題（提出）


提出するもの：絵2枚（画像ファイル）。今日の重要概念の、統計とPythonを1枚ずつです。提出フォームから出してください。期限はありません。

コードは書きません。上手な絵である必要はありません。紙に手で描いて写真を撮っても、描画アプリで描いてもかまいません。


### 今日の重要概念（2つ）

- 統計：p値。差が無いとしたときに、実際の差と同じかそれ以上の差が、偶然に出る割合。差の大きさではないので、区間と組で読む（2・4節）
- Python：乱数で取り直して、くり返す。1回分の「取り直して測る」を`for`でくり返し、リストにためると、偶然のばらつきの分布ができる（2・4節）


### 今日の2枚（絵にする）

2つの重要概念を、1枚ずつ絵にしてください。その考えが1枚で伝わる絵です。数式やコードは描かなくてかまいません。半年後の自分が見て、何の話だったか思い出せる絵を目指します。


<details><summary>詰まったら</summary>

p値なら、たとえば「くじ引きの箱」を描けます。
当たりの札とはずれの札が、AとBに関係なく入っています。何度も引き分けて、差の山を作り、実際の差に旗を立てます。旗より外に出た分が、p値です。山の外れに旗があれば「偶然では出にくい」と言えます。2節の、混ぜた差のヒストグラムが手本になります。

</details>


<details><summary>詰まったら</summary>

乱数で取り直して、くり返すなら、たとえば「料理の味見」を描けます。
大きな鍋から小さじですくって味を書き留め、また鍋に戻します。これを何度もくり返すと、すくうたびの味のぶれが分かります。書き留めた味の一覧がリストで、一覧の広がりがばらつきです。4節の「1回ごとに、後ろに1つ足す」の図が手本になります。

</details>


---


## 発展（任意）


### 同じ2つの見出しを、もう一度

TODO①の10代の記事の2つの見出しは、15分前の別のテストでも比べられていました。そのテストは4案で、うち2案（③と④）がTODO①と同じ見出しです。同じ2つの見出しを比べ直すと、同じ答えになるでしょうか。


In [ ]:
e = df[df['テスト'] == '51c3ba03293a6d6074002fe4']
ne = e['表示'].tolist()
ke = e['クリック'].tolist()
print((e['クリック'] / e['表示']).round(4).tolist())
print(round(pval(to01(ke[2], ne[2]), to01(ke[3], ne[3])), 4))


#### 1行ずつ読む

- 1〜3行目：15分前のテストを取り出し、表示とクリックをリストにする
- 4行目：4案のクリック率
- 5行目：③「劣等感を笑いに」と④「男の子たち」のp値（5節の`to01`と`pval`）

15分前のテストでは、③が1.71%、④が1.65%で、差はほぼありません（p=0.83）。あとのテストでは、1.82%と1.16%で有意でした。同じ2つの見出しで、答えが分かれました。1回の有意は、くり返して確かめます。

試すなら、`same_pairs`をまねて、同じ見出しが別のテストにも出てくる組を探し、2回の結果を比べてみましょう。
